# [문제 4-1] 카페 메뉴 도구(Tool) 호출 체인 구현 (LangChain 사용)

이 노트북은 LangChain의 Tool Calling 기능을 활용하여, 다양한 데이터 소스(로컬 DB, 웹, 위키피디아)에서 정보를 검색하고 종합하여 답변하는 카페 메뉴 AI 어시스턴트를 구현합니다.

**학습 목표:**
- `@tool` 데코레이터를 사용하여 사용자 정의 도구 생성하기
- 텍스트 파일을 Chroma 벡터 DB로 구축하고 검색하기
- 서로 다른 용도의 여러 도구를 하나의 LLM에 연결(`bind_tools`)하기
- `@chain` 데코레이터를 사용하여 간단한 도구 호출 워크플로우 구현하기

In [2]:
import os
from dotenv import load_dotenv
from langchain_community.utilities.tavily_search import TAVILY_API_URL

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print(OPENAI_API_KEY[:2])
UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")
print(UPSTAGE_API_KEY[30:])
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
print(TAVILY_API_KEY[:2])

sk
x4
tv


### (3) 벡터 DB 생성 및 테스트 (Chroma 사용)

`cafe_menu.txt` 파일을 로드하고, 각 메뉴 항목을 별도의 `Document`로 분할한 뒤, Upstage 임베딩 모델을 사용하여 벡터로 변환합니다. 변환된 벡터는 **Chroma** 데이터베이스에 저장하여 나중에 `db_search_cafe_func` 도구가 검색할 수 있도록 준비합니다.

In [3]:
import re
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document
from langchain_upstage import UpstageEmbeddings
from langchain_community.vectorstores import Chroma

# --- 1. 문서 로드 ---
loader = TextLoader("../data/cafe_menu.txt", encoding="utf-8")
documents = loader.load()
print("✅ cafe_menu.txt 로드 완료")

# --- 2. 문서 분할 ---
# 각 메뉴 항목을 별도의 Document 객체로 분할하는 함수
def split_menu_items(document):
    pattern = r'(\d+\.\s.*?)(?=\n\n\d+\.|$)'
    menu_items = re.findall(pattern, document.page_content, re.DOTALL)
    menu_documents = []
    for i, item in enumerate(menu_items, 1):
        menu_name = item.split('\n')[0].split('.', 1)[1].strip()
        menu_doc = Document(
            page_content=item.strip(),
            metadata={
                "source": document.metadata['source'],
                "menu_number": i,
                "menu_name": menu_name
            }
        )
        menu_documents.append(menu_doc)
    return menu_documents

menu_docs = split_menu_items(documents[0])
print(f"✅ 총 {len(menu_docs)}개의 메뉴 항목으로 분할 완료")

# --- 3. 임베딩 모델 설정 ---
embeddings_model = UpstageEmbeddings(model="solar-embedding-1-large")
print("✅ Upstage 임베딩 모델 준비 완료")

# --- 4. Chroma 벡터 DB 생성 및 저장 ---
db_dir = "../db/cafe_db"
cafe_db = Chroma.from_documents(
    documents=menu_docs,
    embedding=embeddings_model,
    persist_directory=db_dir
)
print(f"✅ Chroma 벡터 DB 생성 및 저장 완료 (경로: {db_dir})")

# --- 5. 검색기(Retriever) 테스트 ---
retriever = cafe_db.as_retriever(search_kwargs={"k": 2})
retrieved_docs = retriever.invoke("라떼에 대해 알려줘")
print("\n--- 검색기 테스트 결과 ---")
for doc in retrieved_docs:
    print(doc.metadata)

✅ cafe_menu.txt 로드 완료
✅ 총 10개의 메뉴 항목으로 분할 완료
✅ Upstage 임베딩 모델 준비 완료
✅ Chroma 벡터 DB 생성 및 저장 완료 (경로: ../db/cafe_db)

--- 검색기 테스트 결과 ---
{'source': '../data/cafe_menu.txt', 'menu_number': 2, 'menu_name': '카페라떼'}
{'source': '../data/cafe_menu.txt', 'menu_name': '바닐라 라떼', 'menu_number': 4}


In [4]:
# (1) 도구 정의: db_search_cafe_func
from typing import List
from langchain_core.tools import tool

# 미리 생성된 Chroma DB를 로드합니다.
cafe_db = Chroma(
    embedding_function=embeddings_model,
    persist_directory=db_dir
)

@tool
def db_search_cafe_func(query: str) -> List[Document]:
    """
    로컬 카페 메뉴 데이터베이스에서 정보를 검색할 때 사용합니다.
    메뉴의 가격, 재료, 설명 등과 관련된 질문에 유용합니다.
    """
    docs = cafe_db.similarity_search(query, k=4)
    if docs:
        return docs
    return [Document(page_content="관련 메뉴 정보를 찾을 수 없습니다.")]

/var/folders/_b/pqs_x7fd3pl16l4x573_l2th0000gn/T/ipykernel_78247/3137767912.py:6: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  cafe_db = Chroma(


In [5]:
# (2) 도구 정의: tavily_search_func
from langchain_community.tools import TavilySearchResults

@tool
def tavily_search_func(query: str) -> str:
    """
    최신 정보나 데이터베이스에 없는 정보를 인터넷에서 검색할 때 사용합니다.
    예를 들어, 최신 커피 트렌드나 특정 카페의 위치 정보 등에 유용합니다.
    """
    tavily_search = TavilySearchResults(max_results=3)
    docs = tavily_search.invoke(query)

    formatted_docs = "\n---\n".join([
        f'<Document href="{doc["url"]}"/>\n{doc["content"]}\n</Document>'
        for doc in docs
    ])

    if formatted_docs:
        return formatted_docs

    return "관련 정보를 찾을 수 없습니다."

In [6]:
# (3) 도구 정의: wiki_summary (LCEL 체인을 도구로 변환)
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import WikipediaLoader
from langchain_upstage import ChatUpstage
from pydantic import BaseModel, Field
from textwrap import dedent

# WikipediaLoader를 사용하는 함수 정의
def search_wiki_func(input_data: dict) -> List[Document]:
    wiki_loader = WikipediaLoader(query=input_data["query"], load_max_docs=2, lang="ko")
    return wiki_loader.load()

# 요약 프롬프트 템플릿
summary_prompt = ChatPromptTemplate.from_template("다음 텍스트를 간결하게 요약해 주세요:\n\n{context}\n\n요약:")

# LLM 초기화
llm = ChatUpstage(model="solar-pro", temperature=0.1)

# 요약 체인 생성
summary_chain = (
    {"context": RunnableLambda(search_wiki_func)}
    | summary_prompt | llm | StrOutputParser()
)

# 도구 입력 스키마 정의
class WikiSummarySchema(BaseModel):
    query: str = Field(..., description="위키피디아에서 검색할 주제")

# Runnable 체인을 도구로 변환
wiki_summary = summary_chain.as_tool(
    name="wiki_summary",
    description=dedent("""
        일반적인 지식이나 배경 정보가 필요할 때 위키피디아에서 정보를 검색하고 요약합니다.
        예를 들어, 커피의 역사, 음료 제조 방법 등에 유용합니다.
    """),
    args_schema=WikiSummarySchema
)

/var/folders/_b/pqs_x7fd3pl16l4x573_l2th0000gn/T/ipykernel_78247/2491576804.py:32: LangChainBetaWarning: This API is in beta and may change in the future.
  wiki_summary = summary_chain.as_tool(


In [7]:
# (4) LLM에 모든 도구 바인딩

tools = [db_search_cafe_func, tavily_search_func, wiki_summary]
llm_with_tools = llm.bind_tools(tools=tools)

print("✅ 3개의 도구가 LLM에 성공적으로 바인딩되었습니다.")
print(f" - 바인딩된 도구: {[tool.name for tool in tools]}")

✅ 3개의 도구가 LLM에 성공적으로 바인딩되었습니다.
 - 바인딩된 도구: ['db_search_cafe_func', 'tavily_search_func', 'wiki_summary']


In [8]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.runnables import RunnableConfig, chain
from langchain_core.prompts import MessagesPlaceholder

# 에이전트의 생각을 유도하고, 도구 사용을 결정하는 메인 프롬프트
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 카페 메뉴와 음식에 대한 지식을 갖춘 AI 어시스턴트입니다. 사용자의 질문에 답하기 위해 주어진 도구를 적절히 활용하세요."),
    ("human", "{user_input}"),
    MessagesPlaceholder(variable_name="messages"), # 도구 실행 결과를 담을 자리
])

# LLM 체인
llm_chain = prompt | llm_with_tools

# 전체 워크플로우를 정의하는 @chain 함수
@chain
def cafe_tool_chain(user_input: str, config: RunnableConfig):
    # 초기 입력 설정
    input_ = {"user_input": user_input, "messages": []}

    # 1. LLM 호출하여 도구 사용 결정
    ai_msg = llm_chain.invoke(input_, config=config)

    # 2. 도구 실행
    tool_msgs = []
    if ai_msg.tool_calls:
        for tool_call in ai_msg.tool_calls:
            print(f"▶️ 도구 호출: {tool_call['name']}({tool_call['args']})")
            if tool_call["name"] == "db_search_cafe_func":
                tool_output = db_search_cafe_func.invoke(tool_call, config=config)
            elif tool_call["name"] == "tavily_search_func":
                tool_output = tavily_search_func.invoke(tool_call, config=config)
            elif tool_call["name"] == "wiki_summary":
                tool_output = wiki_summary.invoke(tool_call, config=config)

            tool_msgs.append(ToolMessage(content=str(tool_output), tool_call_id=tool_call['id']))

    # 3. 도구 실행 결과를 포함하여 최종 답변 생성
    input_["messages"].extend([ai_msg, *tool_msgs])
    return llm_chain.invoke(input_, config=config)

print("✅ @chain 데코레이터를 사용한 도구 호출 체인 구현 완료!")

✅ @chain 데코레이터를 사용한 도구 호출 체인 구현 완료!


In [11]:
# 테스트 질문 실행
query = "아메리카노의 가격과 특징은 무엇인가요?"
response = cafe_tool_chain.invoke(query)

# 최종 답변 출력
print("\n--- 최종 답변 ---")
print(response.content)

▶️ 도구 호출: db_search_cafe_func({'query': '아메리카노 가격 및 특징'})

--- 최종 답변 ---
아메리카노의 가격과 특징은 다음과 같습니다:

**1. 핫 아메리카노**  
- **가격**: ₩4,500  
- **주요 원료**: 에스프레소 + 뜨거운 물  
- **특징**:  
  - 에스프레소 샷에 뜨거운 물을 희석해 만든 클래식한 블랙 커피  
  - 원두 본연의 풍미를 가장 잘 느낄 수 있음  
  - 깔끔하고 깊은 풍미, 산미와 쓴맛의 조화  
  - 설탕/시럽 추가 가능 (기본 제공 X)  

**2. 아이스 아메리카노**  
- **가격**: ₩4,500  
- **주요 원료**: 에스프레소 + 차가운 물 + 얼음  
- **특징**:  
  - 핫 아메리카노와 동일한 가격이지만 시원한 버전  
  - 더운 날씨에 인기 많음  
  - 원두 풍미는 유지하되 얼음 덕분에 청량감 추가  

> 📌 **참고**: 콜드브루(₩5,000)와 달리 아메리카노는 에스프레소 기반으로 제조되며, 산미가 더 강하고 추출 시간이 짧습니다. 카페라떼(₩5,500)와 비교하면 우유 없이 깔끔한 맛이 특징입니다.  

특정 카페의 메뉴가 아닌 일반적인 정보를 원하시면 추가 설명이 가능합니다!
